In [ ]:
"""
K-Means Clustering — Three Implementations
=============================================
1. Pure Python from scratch   (no numpy, no sklearn — just the algorithm)
2. NumPy vectorized            (same algorithm, fast array operations)
3. scikit-learn                (production-grade reference implementation)

Run this file directly to fit all three on the same synthetic dataset
and compare their results.
"""

import math
import random



def euclidean_distance(a, b):
    """Straight-line distance between two points (plain lists/tuples)."""
    return math.sqrt(sum((ai - bi) ** 2 for ai, bi in zip(a, b)))


def kmeans_scratch(data, k, max_iters=100, tol=1e-4, seed=42):
    """
    K-means using only core Python — this is the algorithm with nothing
    hidden behind a library call.

    data : list of points, each point a list/tuple of floats
    k    : number of clusters
    returns: (centroids, labels)
    """
    rng = random.Random(seed)
    n_features = len(data[0])

    # --- Initialization: pick k random data points as starting centroids ---
    centroids = [list(p) for p in rng.sample(data, k)]
    labels = [0] * len(data)

    for iteration in range(max_iters):
        # --- Assignment step: give each point to its nearest centroid ---
        for i, point in enumerate(data):
            distances = [euclidean_distance(point, c) for c in centroids]
            labels[i] = distances.index(min(distances))

        # --- Update step: move each centroid to the mean of its members ---
        new_centroids = []
        for cluster_idx in range(k):
            members = [data[i] for i in range(len(data)) if labels[i] == cluster_idx]
            if not members:
                new_centroids.append(centroids[cluster_idx])  # empty cluster: keep old spot
                continue
            mean_point = [
                sum(m[feat] for m in members) / len(members)
                for feat in range(n_features)
            ]
            new_centroids.append(mean_point)

        # --- Convergence check: how far did the centroids move? ---
        shift = sum(euclidean_distance(centroids[i], new_centroids[i]) for i in range(k))
        centroids = new_centroids
        if shift < tol:
            print(f"[scratch]  converged after {iteration + 1} iterations")
            break

    return centroids, labels


def inertia_scratch(data, centroids, labels):
    """Sum of squared distances from each point to its assigned centroid (= J)."""
    return sum(
        euclidean_distance(data[i], centroids[labels[i]]) ** 2
        for i in range(len(data))
    )




import numpy as np


def kmeans_numpy(X, k, max_iters=100, tol=1e-4, seed=42):
    """
    Same algorithm as above, but using NumPy array broadcasting instead
    of Python loops — this is how you'd actually want to run k-means on
    any dataset bigger than a toy example.

    X : ndarray of shape (n_samples, n_features)
    returns: (centroids, labels)
    """
    rng = np.random.default_rng(seed)
    n_samples = X.shape[0]

    # --- Initialization: pick k random rows as starting centroids ---
    init_idx = rng.choice(n_samples, size=k, replace=False)
    centroids = X[init_idx].copy()

    for iteration in range(max_iters):
        # --- Assignment step ---
        # distances[i, j] = distance from point i to centroid j, all at once
        distances = np.linalg.norm(X[:, None, :] - centroids[None, :, :], axis=2)
        labels = np.argmin(distances, axis=1)

        # --- Update step ---
        new_centroids = np.array([
            X[labels == j].mean(axis=0) if np.any(labels == j) else centroids[j]
            for j in range(k)
        ])

        # --- Convergence check ---
        shift = np.linalg.norm(new_centroids - centroids)
        centroids = new_centroids
        if shift < tol:
            print(f"[numpy]    converged after {iteration + 1} iterations")
            break

    return centroids, labels


def inertia_numpy(X, centroids, labels):
    return float(np.sum((X - centroids[labels]) ** 2))




from sklearn.cluster import KMeans


def kmeans_sklearn(X, k, seed=42):
    """
    The production version. Two upgrades over what we wrote by hand:
    - init="k-means++"  -> smarter starting centroids (spread out, not pure random)
    - n_init=10         -> runs the whole algorithm 10 times, keeps the best (lowest J)
    """
    model = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=seed)
    model.fit(X)
    print(f"[sklearn]  converged after {model.n_iter_} iterations")
    return model.cluster_centers_, model.labels_, model.inertia_




if __name__ == "__main__":
    from sklearn.datasets import make_blobs

    X, _ = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)
    data_as_lists = X.tolist()
    k = 4

    print("Running k-means three ways on the same 300-point, 4-cluster dataset\n")

    centroids_s, labels_s = kmeans_scratch(data_as_lists, k)
    cost_s = inertia_scratch(data_as_lists, centroids_s, labels_s)
    print(f"  -> final inertia: {cost_s:.2f}\n")

    centroids_n, labels_n = kmeans_numpy(X, k)
    cost_n = inertia_numpy(X, centroids_n, labels_n)
    print(f"  -> final inertia: {cost_n:.2f}\n")

    centroids_k, labels_k, cost_k = kmeans_sklearn(X, k)
    print(f"  -> final inertia: {cost_k:.2f}\n")

    print("All three land on a very similar inertia value —")
    print("small differences come only from random initialization.")


    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    results = [
        ("From scratch (pure Python)", np.array(centroids_s), np.array(labels_s)),
        ("NumPy vectorized", centroids_n, labels_n),
        ("scikit-learn", centroids_k, labels_k),
    ]
    for ax, (title, centroids, labels) in zip(axes, results):
        ax.scatter(X[:, 0], X[:, 1], c=labels, cmap="tab10", s=20, alpha=0.7)
        ax.scatter(centroids[:, 0], centroids[:, 1], c="black", marker="X", s=150)
        ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])

    plt.tight_layout()
    plt.savefig("kmeans_comparison.png", dpi=130)
    print("\nSaved comparison plot to kmeans_comparison.png")

Running k-means three ways on the same 300-point, 4-cluster dataset

[scratch]  converged after 5 iterations
  -> final inertia: 362.47

[numpy]    converged after 3 iterations
  -> final inertia: 362.47

[sklearn]  converged after 3 iterations
  -> final inertia: 362.47

All three land on a very similar inertia value —
small differences come only from random initialization.

Saved comparison plot to kmeans_comparison.png
